# PySpark reference 15: production patterns

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-15-production-patterns.ipynb)

Idempotent and incremental loads with high-water marks, backfills, CDC with deletes, SCD type 2 with one MERGE, quality gates and quarantine, testing with `assertDataFrameEqual`, job structure and a deployment checklist. Practise it in [PySpark practice 15: production patterns](pyspark-practice-15-production-patterns.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. The session loads Delta Lake; `new_path(name)` returns a fresh table location.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the examples write

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import os
import uuid

from delta.tables import DeltaTable
from pyspark.testing.utils import assertSchemaEqual


def new_path(name):
    """A fresh table location under WORK_DIR."""
    return os.path.join(WORK_DIR, f"{name}-{uuid.uuid4().hex[:6]}")


def assert_df_equal(actual, expected):
    """Equal schemas and equal rows as multisets: the core of pyspark.testing.assertDataFrameEqual, without pandas."""
    assertSchemaEqual(actual.schema, expected.schema)
    missing, unexpected = expected.exceptAll(actual).count(), actual.exceptAll(expected).count()
    if missing or unexpected:
        raise AssertionError(f"DataFrames differ: {missing} expected row(s) missing, {unexpected} unexpected row(s)")

## Contents

| section | entries |
| --- | --- |
| [Idempotent loads](#idempotent-loads) | [Write patterns and idempotency](#write-patterns-and-idempotency), [Incremental loads with a high-water mark](#incremental-loads-with-a-high-water-mark) |
| [Change data capture and slowly changing dimensions](#change-data-capture-and-slowly-changing-dimensions) | [Apply a CDC feed with inserts, updates and deletes](#apply-a-cdc-feed-with-inserts-updates-and-deletes), [SCD type 2 with one `MERGE`](#scd-type-2-with-one-merge) |
| [Data quality](#data-quality) | [Quality gates and quarantine](#quality-gates-and-quarantine) |
| [Testing](#testing) | [`assertDataFrameEqual` and `assertSchemaEqual`](#assertdataframeequal-and-assertschemaequal) |
| [Job structure and operations](#job-structure-and-operations) | [A parameterized job](#a-parameterized-job), [Deployment checklist](#deployment-checklist) |

## Idempotent loads

### Write patterns and idempotency

A job is **idempotent** when running it again with the same input leaves the target unchanged. Retries, backfills and
reruns after a bug fix all depend on it.

| pattern | idempotent | handles deletions | how |
| --- | --- | --- | --- |
| append | **no**: a retry duplicates rows | no | `mode("append")` |
| full overwrite | yes | yes | rebuild the whole table; fine while it is small |
| replace partitions | yes | yes, within the partitions | dynamic partition overwrite, Delta `replaceWhere` |
| `MERGE` on a key | yes | with `whenMatchedDelete` / `whenNotMatchedBySourceDelete` | Delta `MERGE` |

**Notes:** make the **run date a parameter**, never `current_date()`, so reruns and backfills compute the same thing.

In [ ]:
target = new_path("daily_revenue")

def load_day(run_date):
    day_rows = (orders.where(F.to_date("order_ts") == F.lit(run_date).cast("date"))
                .groupBy(F.to_date("order_ts").alias("day"), "status").agg(F.sum("amount").alias("revenue")))
    (day_rows.write.format("delta").mode("overwrite").option("replaceWhere", f"day = '{run_date}'")
     .partitionBy("day").save(target) if DeltaTable.isDeltaTable(spark, target)
     else day_rows.write.format("delta").partitionBy("day").save(target))

for d in ["2026-01-05", "2026-01-06", "2026-01-05"]:  # the third call is a rerun
    load_day(d)
spark.read.format("delta").load(target).orderBy("day", "status").show()

### Incremental loads with a high-water mark

An incremental job reads only rows changed since the last successful run. Keep the **high-water mark** (the largest
source `updated_at` or sequence number loaded) in a small control table, read rows above it, write, and only then advance
it.

**Notes**
- Advance the mark **after** the write commits; a failure in between must cause a reprocess, not a gap.
- Use the source's change timestamp or log sequence number, not the load time.
- Sources that can deliver late rows with older timestamps need an overlap window (`updated_at > mark - 1 hour`) plus an
  idempotent `MERGE` to absorb the repeats.

In [ ]:
source = spark.createDataFrame(
    [(1, "paid", datetime(2026, 1, 5, 9)), (2, "placed", datetime(2026, 1, 5, 10)), (3, "paid", datetime(2026, 1, 6, 8))],
    "order_id int, status string, updated_at timestamp")
silver, control = new_path("silver_orders"), new_path("control")
spark.createDataFrame([("orders", datetime(1970, 1, 1))], "source string, high_water timestamp").write.format("delta").save(control)

def incremental_load(src):
    mark = spark.read.format("delta").load(control).where("source = 'orders'").first().high_water
    new_rows = src.where(F.col("updated_at") > F.lit(mark))
    if new_rows.isEmpty():
        return 0
    if DeltaTable.isDeltaTable(spark, silver):
        (DeltaTable.forPath(spark, silver).alias("t").merge(new_rows.alias("s"), "t.order_id = s.order_id")
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        new_rows.write.format("delta").save(silver)
    new_mark = new_rows.agg(F.max("updated_at")).first()[0]
    DeltaTable.forPath(spark, control).update("source = 'orders'", {"high_water": F.lit(new_mark)})
    return new_rows.count()

print("first run loaded:", incremental_load(source))
updates = source.unionByName(spark.createDataFrame([(2, "paid", datetime(2026, 1, 7, 9))], source.schema))
print("second run loaded:", incremental_load(updates), "| rerun loaded:", incremental_load(updates))
spark.read.format("delta").load(silver).orderBy("order_id").show()

## Change data capture and slowly changing dimensions

### Apply a CDC feed with inserts, updates and deletes

CDC feeds (Debezium, DMS, database logs) deliver one row per change with an operation code and a sequence number. To
apply a batch: keep the **latest change per key** by sequence, then `MERGE` with a delete clause for `D` operations.

**Notes:** order by the source sequence (LSN, version), not the ingestion time. Deduplicating first is mandatory: `MERGE`
fails when several source rows match one target row.

In [ ]:
cdc_target = new_path("cdc_orders")
orders.select("order_id", "status", "amount").write.format("delta").save(cdc_target)
changes = spark.createDataFrame(
    [(2, "shipped", D("84.50"), "U", 10), (2, "returned", D("84.50"), "U", 11), (5, None, None, "D", 12), (13, "placed", D("42.00"), "I", 13)],
    "order_id int, status string, amount decimal(12,2), op string, lsn long")
latest = changes.withColumn("rn", F.row_number().over(Window.partitionBy("order_id").orderBy(F.desc("lsn")))).where("rn = 1")
(DeltaTable.forPath(spark, cdc_target).alias("t").merge(latest.alias("s"), "t.order_id = s.order_id")
 .whenMatchedDelete(condition="s.op = 'D'")
 .whenMatchedUpdate(condition="s.op <> 'D'", set={"status": "s.status", "amount": "s.amount"})
 .whenNotMatchedInsert(condition="s.op <> 'D'", values={"order_id": "s.order_id", "status": "s.status", "amount": "s.amount"})
 .execute())
spark.read.format("delta").load(cdc_target).where("order_id IN (2, 5, 13)").orderBy("order_id").show()

### SCD type 2 with one `MERGE`

A type 2 dimension keeps history: one row per version with `valid_from`, `valid_to` and `is_current`. When an attribute
changes, the current row is **expired** and a new current row **inserted**. One `MERGE` can do both by feeding each
changed key twice: once with a null merge key (which never matches, so it inserts the new version) and once with the real
key (which matches and expires the old version).

**Notes:** compare attributes null-safely (`<=>`) so a change to or from null counts; new keys simply insert.

In [ ]:
dim = new_path("dim_customer")
(customers.select("customer_id", "region").withColumn("valid_from", F.lit(datetime(2024, 1, 1)))
 .withColumn("valid_to", F.lit(None).cast("timestamp")).withColumn("is_current", F.lit(True))
 .write.format("delta").save(dim))
incoming = spark.createDataFrame([(1, "West"), (4, "South"), (2, "North"), (9, "East")], "customer_id int, region string")
load_ts = datetime(2026, 1, 15)

current = spark.read.format("delta").load(dim).where("is_current")
changed = incoming.alias("i").join(current.alias("c"), "customer_id").where(~F.col("i.region").eqNullSafe(F.col("c.region"))).select("i.*")
staged = (changed.withColumn("merge_key", F.lit(None).cast("int"))
          .unionByName(incoming.withColumn("merge_key", F.col("customer_id"))))
(DeltaTable.forPath(spark, dim).alias("t")
 .merge(staged.alias("s"), "t.customer_id = s.merge_key AND t.is_current")
 .whenMatchedUpdate(condition="NOT (t.region <=> s.region)", set={"valid_to": F.lit(load_ts), "is_current": F.lit(False)})
 .whenNotMatchedInsert(values={"customer_id": "s.customer_id", "region": "s.region", "valid_from": F.lit(load_ts),
                               "valid_to": F.lit(None).cast("timestamp"), "is_current": F.lit(True)})
 .execute())
spark.read.format("delta").load(dim).where("customer_id IN (1, 2, 4, 9)").orderBy("customer_id", "valid_from").show()

## Data quality

### Quality gates and quarantine

Check data **before** publishing it, and decide per rule whether a failure stops the job or only routes rows aside:

| rule type | example | typical action |
| --- | --- | --- |
| completeness | required columns not null | quarantine the row |
| validity | amount ≥ 0, status in an allowed set | quarantine the row |
| uniqueness | one row per key | fail the job |
| referential integrity | every order has a customer | quarantine or fail, depending on how late dimensions arrive |
| volume | row count within ±50% of yesterday | fail and alert |

**Pattern:** compute a list of rule-failure names per row with `when`, split into `valid` and `quarantine` on whether it is
empty, write both, and record per-rule counts in a metrics table. Raise an exception (`raise ValueError(...)`) when a
blocking rule fails, so the orchestrator marks the run failed.

In [ ]:
batch = orders.unionByName(spark.createDataFrame([(13, 99, "lost", datetime(2026, 1, 14, 9), D("-5.00")), (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00"))], orders.schema))
rules = {
    "negative_amount": F.col("amount") < 0,
    "bad_status": ~F.col("status").isin("placed", "paid", "shipped", "returned"),
    "unknown_customer": F.col("customer_id").isin([r.customer_id for r in customers.select("customer_id").collect()]) == False,  # noqa: E712
}
checked = batch.withColumn("failures", F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in rules.items()])))
valid, quarantine = checked.where(F.size("failures") == 0).drop("failures"), checked.where(F.size("failures") > 0)
quarantine.select("order_id", "failures").show(truncate=False)
duplicates = batch.groupBy("order_id").count().where("count > 1").count()
print("valid:", valid.count(), "| quarantined:", quarantine.count(), "| duplicate keys:", duplicates)
try:
    if duplicates:
        raise ValueError(f"uniqueness check failed: {duplicates} duplicate order_id value(s)")
except ValueError as error:
    print("gate:", error)

## Testing

### `assertDataFrameEqual` and `assertSchemaEqual`

`pyspark.testing.utils.assertDataFrameEqual(actual, expected, checkRowOrder=False, rtol=1e-5, atol=1e-8)` (also exported as `pyspark.testing.assertDataFrameEqual`) compares two
DataFrames (or lists of `Row`s) and raises an `AssertionError` with a readable diff when they differ.
`assertSchemaEqual(actual, expected)` compares schemas.

| parameter | default | description |
| --- | --- | --- |
| `checkRowOrder` | False | compare in order instead of as multisets |
| `rtol` / `atol` | 1e-5 / 1e-8 | tolerances for floating-point values |
| `ignoreNullable` | True | ignore nullability differences in the schema |
| `ignoreColumnOrder` / `ignoreColumnName` | False | relax column matching |

**Use it for:** unit tests of transformation functions in pytest, with small hand-written input and expected DataFrames.
**Compatibility note:** in PySpark 4.0, `assertDataFrameEqual` imports pandas-on-Spark whenever pandas and pyarrow are
installed, and pandas-on-Spark 4.0 does not support pandas 3, so the call fails with an `ImportError`. Until you upgrade
Spark (or pin `pandas<3`), use a small equivalent such as `assert_df_equal` from the setup cell: `assertSchemaEqual`
plus `exceptAll` in both directions. The examples here use it.

A session fixture builds one local `SparkSession` per test run:

```python
@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.master("local[1]").config("spark.sql.shuffle.partitions", "1").getOrCreate()

def test_paid_revenue(spark):
    actual = paid_revenue(spark.createDataFrame([(1, "paid", 10.0), (1, "placed", 5.0)], "customer_id int, status string, amount double"))
    assertDataFrameEqual(actual, spark.createDataFrame([(1, 10.0)], "customer_id int, revenue double"))
```

In [ ]:
def paid_revenue(df):
    return df.where(F.col("status") == "paid").groupBy("customer_id").agg(F.sum("amount").alias("revenue"))

expected = spark.createDataFrame([(1, D("120.00")), (2, D("260.00")), (3, D("310.00")), (5, D("74.00"))], "customer_id int, revenue decimal(22,2)")
assert_df_equal(paid_revenue(orders), expected)
print("paid_revenue: test passed")
try:
    assertSchemaEqual(paid_revenue(orders).schema, spark.createDataFrame([], "customer_id int, revenue double").schema)
except AssertionError as error:
    print("schema test failed as expected:", str(error).splitlines()[0][:80])

## Job structure and operations

### A parameterized job

Production jobs are scripts, not notebooks: an entry point that parses parameters, builds the session (or receives it
from the platform), runs small tested functions, writes idempotently, logs counts, and exits non-zero on failure.

```python
import argparse, logging
from pyspark.sql import SparkSession

log = logging.getLogger("orders_daily")

def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run-date", required=True)            # never current_date()
    p.add_argument("--source", required=True)
    p.add_argument("--target", required=True)
    args = p.parse_args(argv)

    spark = SparkSession.builder.appName(f"orders_daily {args.run_date}").getOrCreate()
    raw = spark.read.format("delta").load(args.source).where(f"ingest_date = '{args.run_date}'")
    clean = raw.transform(drop_bad_rows).transform(add_business_date)
    checks(clean)                                          # raises on blocking failures
    (clean.write.format("delta").mode("overwrite")
     .option("replaceWhere", f"day = '{args.run_date}'").save(args.target))
    log.info("loaded %s rows for %s", clean.count(), args.run_date)

if __name__ == "__main__":
    main()
```

```bash
spark-submit --master yarn --deploy-mode cluster \
  --conf spark.sql.shuffle.partitions=400 --conf spark.sql.adaptive.enabled=true \
  --py-files deps.zip orders_daily.py --run-date 2026-01-14 --source s3://lake/bronze/orders --target s3://lake/silver/orders
```

### Deployment checklist

| area | check |
| --- | --- |
| correctness | idempotent writes; run date as a parameter; reruns produce identical output |
| data quality | blocking and quarantine rules; metrics table; alerts on volume changes |
| schema | explicit read schemas; schema contracts tested; Delta enforcement on |
| testing | unit tests for transformation functions; an end-to-end test on sample data in CI |
| performance | partitioning and file sizes reviewed; broadcast thresholds; AQE on; no driver collects of big data |
| operations | logs with row counts; Spark UI/history server access; retries configured in the orchestrator |
| security | no credentials in code; secrets from the platform's secret store; least-privilege storage access |
| cost | autoscaling or right-sized clusters; small files compacted; old versions vacuumed |

**Next:** practise these entries in [PySpark practice 15: production patterns](pyspark-practice-15-production-patterns.ipynb).